In [47]:
import json
from pathlib import Path
import pandas as pd
import re
import extract as ext
from shutil import move

In [48]:
ROOT = Path("~/code/selector_examenes").expanduser()

In [49]:
form_classified = pd.read_csv("form_classified.csv", encoding="utf-8")
form_emestrada = pd.read_csv("form_emestrada.csv", encoding="utf-8")

In [50]:
form_emestrada["statement_backup"] = form_emestrada.statement

In [51]:
form_emestrada.head()

,year,exam_name,statement,statement_backup
0,2010,química - 2010 - junio,a) Nitrito de plata b) Hidróxido de magnesio c...,a) Nitrito de plata b) Hidróxido de magnesio c...
1,2010,química - 2010 - junio,a) Peróxido de rubidio b) Hidrogenocarbonato d...,a) Peróxido de rubidio b) Hidrogenocarbonato d...
2,2010,química - 2010 - reserva 1,a) Permanganato de bario b) Peróxido de potasi...,a) Permanganato de bario b) Peróxido de potasi...
3,2010,química - 2010 - reserva 1,a) Hidrogenocarbonato de sodio b) Hidróxido de...,a) Hidrogenocarbonato de sodio b) Hidróxido de...
4,2010,química - 2010 - reserva 2,a) Pentafluoruro de antimonio b) Óxido de plom...,a) Pentafluoruro de antimonio b) Óxido de plom...


In [52]:
form_classified.head()

,year,exam_name,exercise_number,topic,exercise_type,statement,exam_option,exam_file_path
0,2010,Quimica - quimica - quimica - junio - quimica ...,1,Formulación,Formulación,1.- Formule o nombre los compuestos siguientes...,Opción A,examenes_quimica/2010_quimica/2010_quimica_qui...
1,2010,Quimica - quimica - quimica - junio - quimica ...,1,Formulación,Formulación,1.- Formule o nombre los compuestos siguientes...,Opción A,examenes_quimica/2010_quimica/2010_quimica_qui...
2,2010,Quimica - quimica - quimica - junio - quimica ...,1,Formulación,Formulación,1.- Formule o nombre los compuestos siguientes...,Opción A,examenes_quimica/2010_quimica/2010_quimica_qui...
3,2010,Quimica - quimica - quimica - junio - quimica ...,1,Formulación,Formulación,1.- Formule o nombre los compuestos siguientes...,Opción A,examenes_quimica/2010_quimica/2010_quimica_qui...
4,2010,Quimica - quimica - quimica - junio - quimica ...,1,Formulación,Formulación,1.- Formule o nombre los compuestos siguientes...,Opción A,examenes_quimica/2010_quimica/2010_quimica_qui...


In [53]:
form_classified.year.aggregate(["min", "max"])

min    2010
max    2026
Name: year, dtype: int64

In [54]:
form_emestrada.year.aggregate(["min", "max"])

min    2010
max    2025
Name: year, dtype: int64

In [55]:
for row in form_emestrada.statement[:3]:
    print(row)

a) Nitrito de plata b) Hidróxido de magnesio c) 1,1 Dicloroetano
a) Peróxido de rubidio b) Hidrogenocarbonato de calcio c) Butanona
a) Permanganato de bario b) Peróxido de potasio c) Triclorometano


In [56]:
for row in form_classified.statement[:3]:
    print(row)

1.- Formule o nombre los compuestos siguientes: a) Nitrito de plata b) Hidróxido de magnesio
c) 1,1-Dicloroetano d) MoO₃ e) Ca₃(PO₄)₂ f) CH₂OHCH₂OH
1.- Formule o nombre los compuestos siguientes: a) Peróxido de rubidio b) Hidrogenocarbonato de calcio
c) Butanona d) BeH₂ e) HClO₄ f) CH₃CONH₂
1.- Formule o nombre los compuestos siguientes: a) Permanganato de bario b) Peróxido de potasio
c) Triclorometano d) HgO e) ZnS f) CH₂=CHCH₃


Limpiar y adaptar los enunciados de formulación de `form_classified`

In [57]:
def extraer_nombrar_compuestos(statement):
    if not pd.isna(statement):
        start_position = statement.find("a)")
        if " d) " in statement:
            end_position = statement.find(" d) ")
        else:
            end_position = -1
        statement = statement[start_position:end_position]
        # unir valencia con Stock
        statement = re.sub(r"\s+(\([IVX]+\))", r"\1", statement)
        statement = statement.replace(";", "").replace("\n", " ").strip()
        #statement = re.sub(r"[^0-9a-záéíóú]", "", statement.lower())
    
    return statement

In [58]:
for row in form_classified.statement[:3]:
    print(extraer_nombrar_compuestos(row))

a) Nitrito de plata b) Hidróxido de magnesio c) 1,1-Dicloroetano
a) Peróxido de rubidio b) Hidrogenocarbonato de calcio c) Butanona
a) Permanganato de bario b) Peróxido de potasio c) Triclorometano


In [59]:
form_classified["statement"] = form_classified.statement.apply(extraer_nombrar_compuestos)

In [60]:
for row in form_classified.statement[:3]:
    print(row)

a) Nitrito de plata b) Hidróxido de magnesio c) 1,1-Dicloroetano
a) Peróxido de rubidio b) Hidrogenocarbonato de calcio c) Butanona
a) Permanganato de bario b) Peróxido de potasio c) Triclorometano


In [64]:
# x: classified, y: emestrada
form_merged = pd.merge(form_classified, form_emestrada, on=["statement", "year"], how="left")

In [68]:
form_merged.shape

(182, 10)

In [66]:
form_merged.isna().sum()

year                 0
exam_name_x          0
exercise_number      0
topic                0
exercise_type        0
statement            0
exam_option         60
exam_file_path       0
exam_name_y         63
statement_backup    63
dtype: int64

In [67]:
form_merged.loc[form_merged.exam_name_y.isna()]

,year,exam_name_x,exercise_number,topic,exercise_type,statement,exam_option,exam_file_path,exam_name_y,statement_backup
0,2010,Quimica - quimica - quimica - junio - quimica ...,1,Formulación,Formulación,a) Nitrito de plata b) Hidróxido de magnesio c...,Opción A,examenes_quimica/2010_quimica/2010_quimica_qui...,NaN,NaN
5,2010,Quimica - quimica - quimica - junio - quimica ...,1,Formulación,Formulación,a) Sulfato de manganeso(II) b) Hidróxido de ce...,Opción A,examenes_quimica/2010_quimica/2010_quimica_qui...,NaN,NaN
7,2010,Quimica - quimica - quimica - sept - quimica -...,1,Formulación,Formulación,a) Cromato de plata b) Óxido de estaño(IV) c) ...,Opción A,examenes_quimica/2010_quimica/2010_quimica_qui...,NaN,NaN
24,2012,Quimica - sel - 2012 - quimica - 1 química - m...,1,Formulación,Formulación,a) Ácido clórico b) Seleniuro de hidrógeno c) ...,Opción A,examenes_quimica/2012_quimica/2012_quimica_sel...,NaN,NaN
28,2012,Quimica - sel - 2012 - quimica - 3 química - m...,1,Formulación,Formulación,a) Ácido fosfórico b) Bromuro de magnesio c) Á...,Opción A,examenes_quimica/2012_quimica/2012_quimica_sel...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
177,2026,Quimica - quimica - extraordinaria - incompati...,4A,Formulación,Formulación orgánica,a) Nombre o formule los siguientes compuestos:...,NaN,examenes_quimica/2026_quimica/2026_quimica_Qui...,NaN,NaN
178,2026,Quimica - quimica - extraordinaria - titular -...,4A,Formulación,Formulación orgánica,a) Nombre o formule los siguientes compuestos:...,NaN,examenes_quimica/2026_quimica/2026_quimica_Qui...,NaN,NaN
179,2026,Quimica - quimica - ordinaria - incidencias - ...,4A,Formulación,Formulación orgánica,a) Nombre o formule los siguientes compuestos:...,NaN,examenes_quimica/2026_quimica/2026_quimica_Qui...,NaN,NaN
180,2026,Quimica - quimica - ordinaria - incompatibilid...,4A,Formulación,Formulación orgánica,a) Nombre o formule los siguientes compuestos:...,NaN,examenes_quimica/2026_quimica/2026_quimica_Qui...,NaN,NaN


In [ ]:
form_merged.columns

Index(['year', 'exam_name_x', 'exercise_number', 'topic', 'exercise_type',
       'statement', 'exam_option', 'exam_file_path', 'exam_name_y'],
      dtype='str')

In [ ]:
form_merged.year.aggregate(["min", "max"])

min    2010
max    2022
Name: year, dtype: int64

In [69]:
form_merged.head(1).T

,0
year,2010
exam_name_x,Quimica - quimica - quimica - junio - quimica ...
exercise_number,1
topic,Formulación
exercise_type,Formulación
statement,a) Nitrito de plata b) Hidróxido de magnesio c...
exam_option,Opción A
exam_file_path,examenes_quimica/2010_quimica/2010_quimica_qui...
exam_name_y,NaN
statement_backup,NaN


In [ ]:
cols_to_keep = ["year", "exam_file_path", "exam_name_y"]
form_merged = form_merged.drop([col for col in form_merged.columns if col not in cols_to_keep], axis = 1)

In [ ]:
form_merged = form_merged.rename(columns = {"exam_name_y" : "filename_emestrada",
                     "exam_file_path" : "source_file_path",})

In [ ]:
form_merged.head()

,year,source_file_path,filename_emestrada
0,2010,examenes_quimica/2010_quimica/2010_quimica_qui...,química - 2010 - junio
1,2010,examenes_quimica/2010_quimica/2010_quimica_qui...,química - 2010 - reserva 1
2,2010,examenes_quimica/2010_quimica/2010_quimica_qui...,química - 2010 - reserva 1
3,2010,examenes_quimica/2010_quimica/2010_quimica_qui...,química - 2010 - reserva 2
4,2010,examenes_quimica/2010_quimica/2010_quimica_qui...,química - 2010 - septiembre


In [ ]:
form_merged = form_merged.drop_duplicates()

## Renombrar archivos

In [ ]:
# for idx, row in form_merged.iterrows():
#     old = ROOT / row.source_file_path
#     new = old.parent / f"{row.filename_emestrada}.pdf"

#     #print(f"{old} --> {new}")
#     try:
#         move(old, new)
#     except Exception as e:
#         print(f"Archivo: {old.name}. Error: {e}")

In [ ]:
form_emestrada.loc[form_emestrada.year == 2020].sort_values("statement").statement

156    a) Fluoruro de estroncio b) Sulfito de amonio ...
163    a) Fluoruro de plata b) Ácido nitroso c) p-met...
159    a) Hidróxido de cesio b) Permanganato de litio...
160    a) Hidróxido de vanadio(V) b) Bromato de alumi...
155    a) Peróxido de estroncio b) Nitrato de magnesi...
165    a) Peróxido de hidrógeno b) Arsenato de hierro...
154    a) Sulfuro de hidrógeno b) Perclorato de cromo...
164    a) Sulfuro de plomo(II) b) Hidróxido de berili...
161    a) Óxido de aluminio b) Ácido cloroso c) But-2...
162    a) Óxido de molibdeno(IV) b) Sulfato de mangan...
158    a) Óxido de platino(IV) b) Perclorato de potas...
157    a) Óxido de plomo(II) b) Ácido bórico c) o-Bro...
Name: statement, dtype: str

In [87]:
form_classified.loc[form_classified.year == 2025].sort_values("statement").statement

170    a) CH₂=CH―CH₂OH b) CH₃―C(CH₃)₂―C == C H c) Pro...
172        a) CH₂=CH―CH₂―NH₂ b) CH₃―O―CH₃ c) Etilbenceno
173    a) CH₃―C=CH―CH₃ b) CH₃―CH₂―CH(OH)―CH₃ c) Etano...
174    a) CH₃―CH=CH―CH=CH₂ b) CH₃―CH(OH)―CH₂―CN c) 3–...
171    a) CH₃―CH₂―CH₂―CH₂―C―NH₂ b) C H₃― ―OH c) Butan...
175      a) CH₃―CH₂―O―CH₃ b) CH₃―CH―CH₂―CHO c) Etanamina
Name: statement, dtype: str

In [86]:
form_classified.loc[form_classified.year == 2025, "statement"]

170    a) CH₂=CH―CH₂OH b) CH₃―C(CH₃)₂―C == C H c) Pro...
171    a) CH₃―CH₂―CH₂―CH₂―C―NH₂ b) C H₃― ―OH c) Butan...
172        a) CH₂=CH―CH₂―NH₂ b) CH₃―O―CH₃ c) Etilbenceno
173    a) CH₃―C=CH―CH₃ b) CH₃―CH₂―CH(OH)―CH₃ c) Etano...
174    a) CH₃―CH=CH―CH=CH₂ b) CH₃―CH(OH)―CH₂―CN c) 3–...
175      a) CH₃―CH₂―O―CH₃ b) CH₃―CH―CH₂―CHO c) Etanamina
Name: statement, dtype: str

In [75]:
form_classified.year.unique()

array([2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020,
       2021, 2022, 2023, 2025, 2026])